# Assemble multiome data as h5ad objects

In [ ]:
import sys
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)

import json
import plotly.io as pio
pio.renderers.default = 'iframe_connected'

import ArchR_h5ad
import snapatac2 as snap
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
import marsilea as ma
import marsilea.plotter as mp
from pathlib import Path

# Global Scanpy settings
sc.settings.verbosity = 2               # Show progress
sc.settings.set_figure_params(
    dpi=150,                            # High-res output
    dpi_save=300,                       # High-res when saving
    format='pdf',                       # or 'pdf', 'svg'
    facecolor='white',                  # or 'none' for transparent
    frameon=False,                      # No outer frame
    vector_friendly=True,               # No rasterization warnings
    fontsize=10,                        # Base font size
    figsize=(4, 4),                     # Default figure size
    transparent=True                    # Transparent background if saving
)

mpl.rcParams.update({
    "svg.fonttype": "none",       
    "pdf.fonttype": 42,           
    "legend.fontsize": 6,
    "axes.titlesize": 6,
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
})

# Load ATAC data 

In [ ]:
archr_h5ad_export_dir = Path("/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/atac/archr/Multiome_v3/EEC_multiome/h5ad_export")

In [ ]:
adata = sc.read_mtx(archr_h5ad_export_dir / "peak_matrix.mtx").T
adata.obs = pd.read_csv(archr_h5ad_export_dir / "cell_metadata.tsv", index_col=0, sep="\t")
adata.var_names = pd.read_csv(archr_h5ad_export_dir / "features.tsv", index_col=0, sep="\t", header=None).index.tolist()

In [ ]:
# Write to output directory
output_dir = Path("/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/grn")
output_dir.mkdir(parents=True, exist_ok=True)

# Save as h5ad
adata.write_h5ad(output_dir / "EEC_multiome_atac.h5ad")

In [ ]:
chromvar_motif_matrix = pd.read_csv(archr_h5ad_export_dir / "imputed_motif_accessibility_matrix.csv", index_col=0)

adata = ad.AnnData(X=chromvar_motif_matrix.T.values, 
                   obs=pd.DataFrame(index=chromvar_motif_matrix.columns), 
                   var=pd.DataFrame(index=chromvar_motif_matrix.index))

In [ ]:
# Write to output directory
output_dir = Path("/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/grn")
output_dir.mkdir(parents=True, exist_ok=True)

# Save as h5ad
adata.write_h5ad(output_dir / "EEC_multiome_chromvar.h5ad")

## Load RNA data

In [ ]:
adata = sc.read("/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/processed/10X.multiome.processed.controls.integrated.h5ad")

In [ ]:
# Write to output directory
output_dir = Path("/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/grn")
output_dir.mkdir(parents=True, exist_ok=True)

# Save as h5ad
adata.write_h5ad(output_dir / "EEC_multiome_rna.h5ad")

## Load both RNA & ATAC match barcodes

In [ ]:
import re

def normalize_barcode(bc):

    match = re.match(r"^(.*)-(1|2)_(\d+)-(.*)$", bc)
    if match:
        barcode, lane, modal, sample = match.groups()
        return f"{sample}#{barcode}-{lane}"

    match = re.match(r"^(.*)-(1|2)-(.*)$", bc)
    if match:
        barcode, lane, sample = match.groups()
        return f"{sample}#{barcode}-{lane}"

    match = re.match(r"^(.*)-(1|2)$", bc)
    if match:
        barcode, lane = match.groups()
        return f"{barcode}-{lane}"

    raise ValueError(f"Unrecognized barcode format: {bc}")

In [ ]:
adata_rna = sc.read(output_dir / "EEC_multiome_rna.h5ad")
adata_atac = sc.read(output_dir / "EEC_multiome_atac.h5ad")
adata_chromvar = sc.read(output_dir / "EEC_multiome_chromvar.h5ad")

In [ ]:
adata_rna.obs_names = [normalize_barcode(b) for b in adata_rna.obs_names.tolist()]
adata_atac.obs_names = adata_atac.obs.cell_id.tolist()

In [ ]:
keep_barcodes = list(set(adata_rna.obs_names).intersection(set(adata_atac.obs_names)))
keep_barcodes = list(set(keep_barcodes).intersection(set(adata_chromvar.obs_names)))
len(keep_barcodes)

In [ ]:
adata_rna = adata_rna[keep_barcodes, :].copy()
adata_atac = adata_atac[keep_barcodes, :].copy()
adata_chromvar = adata_chromvar[keep_barcodes, :].copy()

In [ ]:
adata_rna.write(output_dir / "EEC_multiome_rna_matched.h5ad")
adata_atac.write(output_dir / "EEC_multiome_atac_matched.h5ad")
adata_chromvar.write(output_dir / "EEC_multiome_chromvar_matched.h5ad")